# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [SELOMR HLODZE]
**Student ID:** [96062028]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [9]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.environ.get("GROQ_API_KEY")

# --- Google Colab (Secrets panel) — uncomment if running in Colab instead ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

assert API_KEY, (
    "Set GROQ_API_KEY in a local .env file (Part 0) or Colab Secrets before running."
)

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",  # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"  # or your provider's model name

print("Client ready.")


Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [ ]:
def ask_llm(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500,
):
    """Reusable wrapper around a single chat-completion call."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response, response.choices[0].message.content


# Test call
resp, answer = ask_llm("In one sentence, what does a microfinance institution do?")
print("ANSWER:\n", answer)
print("\nUSAGE:", resp.usage)


ANSWER:
 A microfinance institution provides small loans, savings, and other financial services to low-income individuals or groups who lack access to traditional banking services, helping them to start or expand small businesses, manage finances, and improve their economic well-being.

USAGE: CompletionUsage(completion_tokens=48, prompt_tokens=53, total_tokens=101, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.051718174, prompt_time=0.001764821, completion_time=0.164076792, total_time=0.165841613)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** 
1. The System role determins the behaviour the model is to use to answer. 
The user role refers to the actual task or question the the user wants the model to answer

2. A token is a small unit of text that an LLM processes. Billing is done per token rather than requests becouse different requests require different amount of compututaions. So by billing by tokens the amount changed is linked to the amount of resources used.

### Part 1.2 — Temperature: the randomness dial

In [15]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

question = "Suggest ONE name for a savings product for market traders in Accra. Give only the name."

results = {0.0: [], 1.2: []}

for temp in [0.0, 1.2]:
    for i in range(5):
        _, answer = ask_llm(question, temperature=temp, max_tokens=60)
        results[temp].append(answer)


# TODO: Print all 10 answers, grouped by temperature.

for temp, answers in results.items():
    print(f"\n=== temperature = {temp} ===")
    for a in answers:
        print(a)


=== temperature = 0.0 ===
MakolaSave
MakolaSave
MakolaSave
MakolaSave
MakolaSave

=== temperature = 1.2 ===
MakolaSave
Makola Save
Makola Save
MakolaSave
Makola Plus


**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** The reponses are the same at temperature 0 but at temperature 1.2 the model answers start to differ.
A low temperature will be more appropriate because the model is expected to give similar responses for the same input when it comes to giving out loans. there should be some form of consistency. 

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [16]:
LETTERS = {
    "L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",
    "L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",
    "L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",
    "L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",
    "L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",
    "L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
    "L001": {
        "applicant_name": "Akosua Mensah",
        "amount_ghs": 8000,
        "purpose": "buy deep freezer / expand into frozen foods",
        "monthly_profit_ghs": 900,
        "has_collateral_or_guarantor": True,
        "repayment_months": 20,
    },
    "L003": {
        "applicant_name": "Efua Darko",
        "amount_ghs": 15000,
        "purpose": "industrial sewing machines and fabric stock",
        "monthly_profit_ghs": 2800,
        "has_collateral_or_guarantor": True,
        "repayment_months": 15,
    },
    "L006": {
        "applicant_name": "Kofi",
        "amount_ghs": 50000,
        "purpose": "car wash, provision shop, phone imports",
        "monthly_profit_ghs": None,
        "has_collateral_or_guarantor": False,
        "repayment_months": 12,
    },
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [ ]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this:\n\n{letter}"

for lid in ["L002", "L006"]:
    _, out = ask_llm(SUMMARY_PROMPT_V1.format(letter=LETTERS[lid]), temperature=0.7)
    print(f"V1 summary of {lid} \n{out}\n")


# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SUMMARY_SYSTEM_V2 = (
    "You are an assistant to a microfinance loan officer in Ghana. You write short, "
    "strictly factual briefs of loan application letters. Rules:  only state facts "
    "explicitly present in the letter, never invent or infer numbers, dates, or "
    "qualifications that are not written,  stay neutral in tone — do not praise or "
    "criticize the applicant,  write 3-4 sentences,"
)
SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter}"


def summarize(letter_text, temperature=0.0):
    _, out = ask_llm(
        SUMMARY_PROMPT_V2.format(letter=letter_text),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=temperature,
        max_tokens=200,
    )
    return out


# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
for lid in ["L002", "L006"]:
    out = summarize(LETTERS[lid])
    print(f" V2 summary of {lid} \n{out}\n")


--- V1 summary of L002 ---
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's struggling due to slow business, but expects it to improve after the festive season. He has no collateral, but promises to repay the loan as soon as possible.

--- V1 summary of L006 ---
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be business-minded and trustworthy. He promises to repay the loan within one year, once his businesses are successful, but has no collateral to offer.

 V2 summary of L002 
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan. He is requesting GHS 25,000 to repair his trotro engine and settle personal debts. Mr. Boateng does not currently have collateral to offer. He expects to repay the loan when he is able to do so.

 V2

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** 
1.V1 made unsupported inferences eg for L006 it says “He has no prior experience” while the text only says “has not started any of these yet.” v2 doesnt assume he has no prior experience and only says "has not yet initiated any of these business ventures".
   V1 used judgmental language. For example, for L002 it says “He’s struggling due to slow business” while the text only says “Business has been slow.” V2 avoids this interpretation and only states the facts given in the application.
V2 fixed V1's tendency to infer information and use interpretive language by explicitly requiring factual, neutral summaries with no invented details.
   2.v1 could invent some details that are not present in the application, this could have an impeact int he loan decisions. and example of this is if the model assumes that the applicant has collateral when the letter doesnt say so.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [ ]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

import json as _json
import re
import pandas as pd

FEWSHOT_LETTER = (
    "Dear Sir, my name is Kofi Oppong, a shoe repairer in Tema. I request GHS 9,000 "
    "to buy leather. My shop usually makes about GHS 800 profit monthly. I do not have a "
    "guarantor yet."
)

FEWSHOT_JSON = {
    "applicant_name": "Kofi Oppong",
    "amount_ghs": 9000,
    "purpose": "buy leather",
    "monthly_profit_ghs": 800,
    "has_collateral_or_guarantor": False,
    "repayment_months": None,
}

EXTRACT_SYSTEM = (
    "You extract structured data from microfinance loan application letters. "
    "Return ONLY a single JSON object — no prose, no markdown code fences, no explanation. "
    "The JSON object must have EXACTLY these keys:\n"
    "  applicant_name (string), amount_ghs (number), purpose (string),\n"
    "  monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),\n"
    "  repayment_months (number or null).\n"
    "If a field is not explicitly stated in the letter, use null. "
    "For has_collateral_or_guarantor, use true if a collateral or guarantor is explicitly "
    "stated, and false if the letter explicitly states that there is none. "
    "Do not guess or infer numeric values that are not written in the text."
)

EXTRACT_PROMPT = """Example letter:{fewshot_letter}
Example output:{fewshot_json}
Now extract the same fields from this letter. Return ONLY the JSON object.
Letter:{letter}"""


# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).


def extract_fields(letter_text, temperature=0.0):
    prompt = EXTRACT_PROMPT.format(
        fewshot_letter=FEWSHOT_LETTER,
        fewshot_json=_json.dumps(FEWSHOT_JSON),
        letter=letter_text,
    )

    _, raw = ask_llm(
        prompt,
        system_prompt=EXTRACT_SYSTEM,
        temperature=temperature,
        max_tokens=300,
    )

    cleaned = re.sub(r"^```(json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()

    try:
        return _json.loads(cleaned)
    except _json.JSONDecodeError:
        print(f"WARNING: could not parse JSON for this letter. Raw output was:\n{raw}")
        return None


# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

rows = []

for lid, text in LETTERS.items():
    fields = extract_fields(text)

    row = {"letter_id": lid}
    row.update(fields if fields else {})
    rows.append(row)

extraction_df = pd.DataFrame(rows).set_index("letter_id")

extraction_df

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
letter_id,,,,,,
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** 1. The model will just memorise information from the example instead of learning the pattern. Using a separate example tests whether the prompt works on unseen letters    2. frabicated numbers or answers based of the other information given.      3. Because we want consistent and predictable results when converting the same text into structured data but for creative tasks a higher temperature can be used to produce varying creative outputs. 


### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [ ]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_SYSTEM = (
    "You are a decision-support assistant to a microfinance loan officer in Ghana. "
    "You do not approve or reject a loan yourself  "
    "EVERY FINAL DECISION IS MADE BY A HUMAN OFFICER. "
    "You produce a structured brief with exactly four sections: "
    "1) Strengths, 2) Risks / red flags, 3) Missing information the officer should request, "
    "4) Suggested next step (choose from: 'invite for interview', 'request documents', "
    "'flag for senior review', 'proceed to standard review'). "
    "Base every point only on the letter and the extracted data provided to you; do not "
    "create your own facts."
)


# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
BRIEF_PROMPT = """Loan letter:{letter}
Extracted data:{extracted}
Produce the four-section brief described in your instructions."""


def make_brief(letter_text, extracted_fields, temperature=0.0):
    prompt = BRIEF_PROMPT.format(
        letter=letter_text, extracted=_json.dumps(extracted_fields)
    )
    _, out = ask_llm(
        prompt, system_prompt=BRIEF_SYSTEM, temperature=temperature, max_tokens=400
    )
    return out


briefs = {}
for lid, text in LETTERS.items():
    extracted = extraction_df.loc[lid].to_dict() if lid in extraction_df.index else {}
    briefs[lid] = make_brief(text, extracted)

for lid in ["L001", "L002", "L006"]:
    print(f"\n Brief for {lid} \n{briefs[lid]}")



 Brief for L001 
## 1. Strengths
- The applicant, Akosua Mensah, has a long-standing business experience of 12 years selling provisions at Makola Market.
- She has a stable monthly profit of GHS 900, indicating a consistent income stream.
- Akosua has demonstrated savings discipline through the susu scheme, accumulating GHS 2,500 over two years without missing a contribution.
- She has a guarantor, her sister, who is a teacher, potentially providing an additional layer of financial security.
- The proposed monthly repayment of GHS 450 over 20 months is roughly half of her monthly profit, suggesting a manageable repayment plan.

## 2. Risks / red flags
- The loan amount of GHS 8,000 is significant compared to her monthly profit and savings, which might pose a risk if her business does not expand as planned.
- There is no detailed information provided about the sister's financial stability or ability to act as a guarantor effectively.
- The expansion into frozen foods introduces new ris

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**1. L003 was correctly identified as the stronger one due to its its registered business, GHS 2,800 monthly profit, collateral, and sales records while L006 was correctly identified as weaker due to no existing businesses, no collateral, and no stated profit. But, the system wrongly assumed that Kofi lacks experience because the letter says he hasn't started the businesses yet. 2. On the ethical side, it is not right to make a decision that has alot on impact on a human life be decided solely by a model that can be biased and is not 100% accurate. And on a practical standpoint the model is not allowed to say approve or reject because humans can tend to get lazy in the confirmation process and just accept whatever the model decides.


### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 5aac91f

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).
fields_to_check = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months",
]


def values_match(field, pred, gold):
    if gold is None:
        return pred is None
    if field == "applicant_name" and isinstance(pred, str) and isinstance(gold, str):
        return pred.strip().lower() == gold.strip().lower()
    if field == "purpose":
        # purpose is free text — treat as correct if the gold keywords all appear
        if not isinstance(pred, str):
            return False
        pred_l = pred.lower()
        return (
            all(w in pred_l for w in gold.lower().split() if len(w) > 3)
            or pred_l.strip() == gold.lower().strip()
        )
    return pred == gold


# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

table = {}
for lid, gold in GOLD.items():
    pred = extraction_df.loc[lid].to_dict() if lid in extraction_df.index else {}
    table[lid] = {f: values_match(f, pred.get(f), gold[f]) for f in fields_to_check}

acc_df = pd.DataFrame(table)
acc_df["accuracy"] = acc_df.mean(axis=1)
acc_df

,L001,L003,L006,accuracy
applicant_name,True,True,True,1.000000
amount_ghs,True,True,True,1.000000
purpose,True,True,False,0.666667
monthly_profit_ghs,True,True,False,0.666667
has_collateral_or_guarantor,True,True,True,1.000000
repayment_months,True,True,True,1.000000


### Part 4.2 — Reliability: is the system consistent?

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

reliability = {0.0: [], 1.0: []}

for temp in [0.0, 1.0]:
    for i in range(5):
        result = extract_fields(LETTERS["L004"], temperature=temp)
        reliability[temp].append(result)


# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

for temp, results in reliability.items():
    valid_json = sum(r is not None for r in results)
    serialized = [_json.dumps(r, sort_keys=True) for r in results if r is not None]
    unique_count = len(set(serialized))

    print(
        f"temperature={temp}: "
        f"{valid_json}/5 valid JSON, "
        f"{unique_count} unique result(s) among the valid ones"
    )

temperature=0.0: 5/5 valid JSON, 1 unique result(s) among the valid ones
temperature=1.0: 5/5 valid JSON, 2 unique result(s) among the valid ones


### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# Test 1 — ask about a detail NOT present in a letter
test1_prompt = (
    "Based on this letter, what is the applicant's credit score?\n\n" + LETTERS["L001"]
)
_, test1_out = ask_llm(
    test1_prompt,
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0.0,
)
print("TEST 1 output:\n", test1_out)
print(
    "\nVerdict: PASS if it states the letter does not mention a credit score; "
    "FAIL if it states or estimates a number."
)


# Test 2 — feed the extractor an irrelevant text
weather_text = (
    "Accra will see scattered showers this afternoon with a high of 29C "
    "and light winds from the southwest."
)

test2_out = extract_fields(weather_text)
print("\nTEST 2 output:\n", test2_out)
print(
    "\nVerdict: PASS if applicant_name/amount/etc. are null/false and no applicant is "
    "invented; FAIL if it fabricates a name or numbers."
)


# TODO: Record the outputs verbatim below and label each PASS or FAIL.
# TODO: Record the outputs verbatim below and label each PASS or FAIL.

# TEST 1 OUTPUT :
# The letter does not mention the applicant's credit score. Akosua Mensah has been selling
# provisions at Makola Market for 12 years and has a history of saving with the susu scheme,
# having saved GHS 2,500 over two years without missing a contribution. She is applying for a
# loan of GHS 8,000 to expand her business. Her proposed monthly repayment amount is GHS 450
# over 20 months.

# VERDICT: PASS


# TEST 2 OUTPUT :
# {'applicant_name': None, 'amount_ghs': None, 'purpose': None,
#  'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None,
#  'repayment_months': None}

# VERDICT: PASS

TEST 1 output:
 The letter does not mention the applicant's credit score. Akosua Mensah has been selling provisions at Makola Market for 12 years and has a history of saving with the susu scheme, having saved GHS 2,500 over two years without missing a contribution. She is applying for a loan of GHS 8,000 to expand her business. Her proposed monthly repayment amount is GHS 450 over 20 months.

Verdict: PASS if it states the letter does not mention a credit score; FAIL if it states or estimates a number.

TEST 2 output:
 {'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}

Verdict: PASS if applicant_name/amount/etc. are null/false and no applicant is invented; FAIL if it fabricates a name or numbers.


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** 1.The extraction accuracy was high. The purpose field was the hardest because it is free text. 2.that temperature 0 produced more consistent results, while a higher temperature can produce more variation. 3.No, my system did not hallucinate under probing.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** 1. applicants who write poorly in English but run solid businesses will be evaluated unfairly becasue the model might mis interpret some aspects of their application. 2. it could pose privacy and secuirity risks. I would check the SLA the API proposes and evaluate it to see if it aligns with our values. 3.I will make human review mandatory for every loan decison. I will log every action to check for any irregularity.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** 1. similarity: both involve tuning inputs to improve performance. difference: hyperparameters controls how the model learns while prompt engineering changes the instruction giving to the model. 2. I will not trust the model to runa utonomously becous of the impact of the decisons to be made.The most impressive results recieved still made mistakes so i cant trust a delicate task to it. 3.

---
### Submission checklist

- [y ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ y] **No API key anywhere in the notebook or the commit history.**
- [ y] Every **Student Reasoning** box is filled in with full sentences.
- [ y] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ y] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ y] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ y] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.